In [1]:
import os
gpu_ids = [0]
os.environ["CUDA_VISIBLE_DEVICES"] = ",".join(map(str, gpu_ids))
import torch

In [ ]:
import os
import pandas as pd
import numpy as np
from PIL import Image, ImageColor
import cv2

def recolor_multiple_examples_lab(
    csv_path,
    image_dir,
    mask_dir,
    save_dir,
    target_colors=("Red", "Green", "Yellow"),
    output_csv_path="metadata_augmented_lab.csv",
    max_images=1900
):
    df = pd.read_csv(csv_path)
    rows = df

    os.makedirs(save_dir, exist_ok=True)
    new_rows = []

    for _, row in rows.iterrows():
        image_id = str(row["id"])
        image_path = os.path.join(image_dir, f"{image_id}.jpg")
        mask_path = os.path.join(mask_dir, f"{image_id}_mask.png") 

        if not os.path.exists(image_path):
            print(f"❌ Missing image for {image_id}")
            continue
        if not os.path.exists(mask_path):
            print(f"❌ Missing mask for {image_id}")
            continue

        # Load image and convert to LAB
        image_bgr = cv2.imread(image_path)
        image_lab = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2LAB)
        H, W = image_lab.shape[:2]

        # Load and resize mask
        mask_gray = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        if mask_gray is None:
            print(f"❌ Failed to load mask image: {mask_path}")
            continue

        mask_resized = cv2.resize(mask_gray, (W, H))
        mask_binary = (mask_resized > 127).astype(np.uint8)

        for color in target_colors:
            if color.lower() == row["baseColour"].lower():
                continue

            try:
                target_rgb = np.array(ImageColor.getrgb(color)).astype(np.uint8).reshape(1, 1, 3)
                target_lab = cv2.cvtColor(target_rgb, cv2.COLOR_RGB2LAB)[0, 0]
            except:
                print(f"Invalid color: {color}")
                continue

            # Recolor using LAB blending
            recolored_lab = image_lab.copy()
            recolored_lab[:, :, 1] = np.where(mask_binary, target_lab[1], recolored_lab[:, :, 1])
            recolored_lab[:, :, 2] = np.where(mask_binary, target_lab[2], recolored_lab[:, :, 2])

            recolored_bgr = cv2.cvtColor(recolored_lab, cv2.COLOR_LAB2BGR)
            recolored_rgb = cv2.cvtColor(recolored_bgr, cv2.COLOR_BGR2RGB)

            new_id = f"{image_id}_{color.lower()}"
            new_path = os.path.join(save_dir, f"{new_id}.jpg")
            Image.fromarray(recolored_rgb).save(new_path)

            new_row = row.copy()
            new_row["id"] = new_id
            new_row["baseColour"] = color
            new_rows.append(new_row)

            print(f"✅ Saved recolored image: {new_path}")

    # Save CSV
    if new_rows:
        pd.DataFrame(new_rows).to_csv(output_csv_path, index=False)
        print(f"✅ CSV with {len(new_rows)} recolored entries saved to: {output_csv_path}")
    else:
        print("⚠️ No new recolored rows were generated.")


In [11]:
recolor_multiple_examples_lab(
    csv_path="./pure_tshirt.csv",
    image_dir="../fashion-dataset/images",
    mask_dir="../fashion-dataset/masks",
    save_dir="../fashion-dataset/fake_images",  
    target_colors=["Red", "Green", "Yellow", "Orange","Blue", "Indigo", "Violet"]
)

✅ Saved recolored image: ../fashion-dataset/fake_images/4729_red.jpg
✅ Saved recolored image: ../fashion-dataset/fake_images/4729_yellow.jpg
✅ Saved recolored image: ../fashion-dataset/fake_images/4729_orange.jpg
✅ Saved recolored image: ../fashion-dataset/fake_images/4729_blue.jpg
✅ Saved recolored image: ../fashion-dataset/fake_images/4729_indigo.jpg
✅ Saved recolored image: ../fashion-dataset/fake_images/4729_violet.jpg
✅ CSV with 6 recolored entries saved to: metadata_augmented_lab.csv
